In [1]:
import re

In [2]:
STREET_TYPE_MAP = {
    "st":        "rue",
    "str":       "rue",
    "street":    "rue",
    "rue":       "rue",
    "av":        "avenue",
    "ave":       "avenue",
    "avenue":    "avenue",
    "blvd":      "boulevard",
    "boul":      "boulevard",
    "boulevard": "boulevard",
    "ch":        "chemin",
    "chemin":    "chemin",
    "pl":        "place",
    "place":     "place",
    "cres":      "croissant",
    "croissant": "croissant",
    "dr":        "drive",
    "drive":     "drive",
    "rd":        "road",
    "road":      "road",
    "crt":       "court",
    "court":     "court",
    "terr":      "terrasse",
    "terrasse":  "terrasse",
    "montee":    "montée",
    "montée":    "montée",
    "rang":      "rang",
    "rte":       "route",
    "route":     "route",
}

STREET_NAME_MAP = {
    "st":        "saint",
    "ste":       "sainte",
    "saint":     "saint",
    "sainte":    "sainte",
    "mt":        "mont",
    "mont":      "mont",
}

def normalize_street_type(word: str) -> str:
    return STREET_TYPE_MAP.get(word.lower().strip("."), word.lower())

def normalize_street_name(name: str) -> str:
    words = name.lower().split("-")
    return "-".join(STREET_NAME_MAP.get(w, w) for w in words)

def parse_address(raw: str | None) -> dict:
    """
    Parse raw Centris address into structured components.
    e.g. "353 - 355A, 3e Avenue, apt. 304 Montréal (Ville-Marie)"
    """
    if not raw:
        return {}

    result = {
        "civic_number": None,
        "unit_number":  None,
        "street_name":  None,
        "street_type":  None,
        "borough":      None,
        "city":         None,
        "raw_address":  raw.strip(),
    }

    # Extract borough from parentheses — e.g. "Montréal (Ville-Marie)"
    borough_match = re.search(r'\(([^)]+)\)', raw)
    if borough_match:
        result["borough"] = borough_match.group(1).strip()
        raw = raw[:borough_match.start()].strip()

    # Extract city — last word(s) before borough
    city_match = re.search(r',?\s*([A-ZÀ-Ü][a-zà-ü]+(?:\s[A-ZÀ-Ü][a-zà-ü]+)*)$', raw)
    if city_match:
        result["city"] = city_match.group(1).strip()
        raw = raw[:city_match.start()].strip()

    # Extract unit/apt number
    unit_match = re.search(
        r',?\s*(apt\.?|app\.?|unit|#|lot|suite)\s*([\w-]+)',
        raw, re.IGNORECASE
    )
    if unit_match:
        result["unit_number"] = unit_match.group(2).strip()
        raw = raw[:unit_match.start()] + raw[unit_match.end():]

    # Extract civic number — handles "353", "353-355", "9635B", "353 - 355A"
    civic_match = re.match(r'^\s*(\d+\w*(?:\s*-\s*\d+\w*)?)', raw)
    if civic_match:
        result["civic_number"] = re.sub(r'\s+', '', civic_match.group(1))  # normalize spaces in range
        raw = raw[civic_match.end():].strip().lstrip(",").strip()

    # Remaining is street — split into type and name
    # e.g. "3e Avenue", "Rue Saint-Mathieu", "Boulevard LaSalle"
    parts = raw.strip().split()
    if len(parts) >= 2:
        # Check if first word is a street type
        if normalize_street_type(parts[0]) in STREET_TYPE_MAP.values():
            result["street_type"] = normalize_street_type(parts[0])
            result["street_name"] = normalize_street_name(" ".join(parts[1:]))
        # Check if last word is a street type
        elif normalize_street_type(parts[-1]) in STREET_TYPE_MAP.values():
            result["street_type"] = normalize_street_type(parts[-1])
            result["street_name"] = normalize_street_name(" ".join(parts[:-1]))
        else:
            result["street_name"] = normalize_street_name(raw.strip().lower())

    return result

In [5]:
parse_address("1960 rue saint-martin Montréal (Ville-Marie)")

{'civic_number': '1960',
 'unit_number': None,
 'street_name': 'saint-martin',
 'street_type': 'rue',
 'borough': 'Ville-Marie',
 'city': 'Montréal',
 'raw_address': '1960 rue saint-martin Montréal (Ville-Marie)'}

In [6]:

# from jose import jwt, JWTError    
# import models
# import schemas 
# from fastapi import Depends, HTTPException, status
# from fastapi.security import OAuth2PasswordBearer
# from dotenv import load_dotenv, find_dotenv # type: ignore
# import os   
# _ = load_dotenv(find_dotenv())

# oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

# SECRET_KEY = os.getenv("SECRET_KEY") # type: ignore
# ALGORITHM = os.getenv("ALGORITHM") # type: ignore
# ACCESS_TOKEN_EXPIRE_MINUTES = int(os.getenv("ACCESS_TOKEN_EXPIRE_MINUTES")) # type: ignore

# def verify_access_token(token: str, credentials_exception):
#     try:
#         payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
#         id: str = payload.get("user_id")
#         if id is None:
#             raise credentials_exception
#         token_data = schemas.TokenData(id=id)
#     except JWTError:
#         raise credentials_exception
    
#     return token_data